# Module 2 Worksheet — Embeddings & Vector Search
**Corrected in this version:** embedding calls go through `embedder.embed_query()` instead of `get_embedding(text, model=MODEL_JINA)`.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../../wrapper_fix"))      # folder containing the corrected inhouse_wrappers.py
sys.path.append(os.path.abspath("../../inhouse_rag_capstone"))  # folder containing your real inhouse_llm.py

from inhouse_llm import MODEL_QWEN3_14B, MODEL_QWEN3_30B, MODEL_MISTRAL, MODEL_LLAMA, MODEL_DEVSTRAL, MODEL_QWEN2_5_VL_7B
from inhouse_wrappers import get_chat_model, InHouseEmbeddings, build_vision_messages, llm_for
from langchain_core.messages import SystemMessage, HumanMessage

embedder = InHouseEmbeddings()

def ask(system_prompt, user_prompt, model=MODEL_QWEN3_14B, max_tokens=500):
    """Correctly-routed replacement for calling multimodal_chat() directly."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]).content

def ask_vision(system_prompt, user_prompt, image_base64, model=MODEL_QWEN2_5_VL_7B, max_tokens=500):
    """Correctly-routed, correctly-formatted multimodal call."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke(build_vision_messages(system_prompt, user_prompt, image_base64)).content

print("Setup OK")

## 1. Distance metrics by hand

In [ ]:
import numpy as np

def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def euclidean(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))

def dot(a, b):
    return float(np.dot(a, b))

v1 = embedder.embed_query("What is RAG?")
v2 = embedder.embed_query("Explain retrieval-augmented generation")
v3 = embedder.embed_query("What is the capital of France?")

for name, vec in [("similar", v2), ("unrelated", v3)]:
    print(f"vs {name}: cosine={cosine(v1, vec):.3f}  euclidean={euclidean(v1, vec):.3f}  dot={dot(v1, vec):.3f}")

## 2. Reproducing this module's teaser bug: metric mismatch
Simulate what happens if vectors are NOT normalized and you use dot product as if it were cosine similarity.

In [ ]:
v1_arr = np.array(v1)
v3_arr = np.array(v3)

# Artificially scale v3 up in magnitude (simulating an un-normalized embedding bug)
v3_scaled = v3_arr * 50

print("True cosine similarity (unrelated pair):", cosine(v1_arr, v3_arr))
print("Dot product, normal scale:", dot(v1_arr, v3_arr))
print("Dot product, v3 artificially scaled up:", dot(v1_arr, v3_scaled))
print("\nNotice: dot product with the scaled vector can EXCEED the dot product\n"
      "of a genuinely more similar but normally-scaled pair — magnitude pollutes\n"
      "the ranking. This is exactly the bug in concept_notes.md's teaser problem.")

## 3. PCA vs t-SNE vs UMAP, side by side
`pip install scikit-learn umap-learn matplotlib --break-system-packages`

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

texts = [
    "RAG combines retrieval and generation.",
    "Retrieval-augmented generation grounds answers in context.",
    "MCP standardizes tool calling for LLMs.",
    "Model Context Protocol lets models call external tools.",
    "12 times 7 equals 84.",
    "Basic arithmetic: multiplication of two integers.",
]
vectors = np.array(embedder.embed_documents(texts))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
pca_coords = PCA(n_components=2).fit_transform(vectors)
axes[0].scatter(pca_coords[:, 0], pca_coords[:, 1])
axes[0].set_title("PCA")

tsne_coords = TSNE(n_components=2, perplexity=2, random_state=0).fit_transform(vectors)
axes[1].scatter(tsne_coords[:, 0], tsne_coords[:, 1])
axes[1].set_title("t-SNE")

for ax, coords in zip(axes, [pca_coords, tsne_coords]):
    for i, t in enumerate(texts):
        ax.annotate(str(i), (coords[i, 0], coords[i, 1]))
plt.show()
for i, t in enumerate(texts):
    print(i, t)

## Teaser exercise
With only 6 points, do PCA and t-SNE actually disagree on which points cluster together? Try it again with 30+ points (add more sample texts) — that's usually where the methods start visibly diverging.